# DBU counter-attack analysis


## 1. Rebuild and load analysis tables

This notebook is intentionally narrow. It rebuilds the CSV files used for the DBU section and then displays the benchmark and feature-profile tables for the men, women and U21 datasets. The heavy model scoring is handled before this notebook; this notebook is mainly for presenting the final analysis tables.


In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

# Run this notebook from the Data_DBU project folder.
BASE_DIR = Path(".").resolve()
ANALYSIS_DIR = BASE_DIR / "Data" / "derived" / "dbu_mens_hybrid_analysis"
SHOT_PROFILE_DIR = ANALYSIS_DIR / "shot_profile"

build_scripts = [
    "build_dbu_mens_hybrid_analysis.py",
    "analyze_dbu_mens_shot_profile.py",
]

for script_name in build_scripts:
    script_path = BASE_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Could not find {script_path}. Run the notebook from the project root.")
    subprocess.run(
        [sys.executable, str(script_path)],
        cwd=BASE_DIR,
        check=True,
        capture_output=True,
        text=True,
    )

print("Analysis CSV files rebuilt.")

DATASETS = {
    "men": "Men H_EURO2024",
    "women": "Women Q_EURO2025",
    "u21": "U21 EURO2025",
}


Analysis CSV files rebuilt.


In [2]:
# Load the report-facing tables produced by the build scripts.
benchmarks_by_dataset = {
    dataset_key: pd.read_csv(ANALYSIS_DIR / f"denmark_vs_hybrid_benchmarks_{dataset_key}.csv")
    for dataset_key in DATASETS
}

feature_profiles_by_dataset = {
    dataset_key: pd.read_csv(SHOT_PROFILE_DIR / f"shot_vs_nonshot_feature_profile_{dataset_key}.csv")
    for dataset_key in DATASETS
}


## 2. Counter-attack benchmark tables

These tables compare Denmark with three reference groups: the tournament average, the selected semi-finalists/finalists, and teams that won their match. The purpose is to see whether Denmark differs mainly in counter-attack volume, shot production, box entries or progression toward goal.


In [3]:
benchmark_cols = [
    "benchmark",
    "matches",
    "hybrid_counterattacks",
    "counterattacks_per_match",
    "shot_rate",
    "box_entry_proxy_rate",
    "avg_progression_goal_m",
]


def make_benchmark_table(dataset_key):
    table = benchmarks_by_dataset[dataset_key][benchmark_cols].copy()
    table["counterattacks_per_match"] = table["counterattacks_per_match"].round(2)
    table["shot_rate"] = (100 * table["shot_rate"]).round(1)
    table["box_entry_proxy_rate"] = (100 * table["box_entry_proxy_rate"]).round(1)
    table["avg_progression_goal_m"] = table["avg_progression_goal_m"].round(1)
    return table.rename(
        columns={
            "benchmark": "Benchmark",
            "matches": "Team-games",
            "hybrid_counterattacks": "Hybrid CAs",
            "counterattacks_per_match": "CAs per team-game",
            "shot_rate": "Shot rate (%)",
            "box_entry_proxy_rate": "Box entry rate (%)",
            "avg_progression_goal_m": "Progression toward goal (m)",
        }
    )


for dataset_key, dataset_name in DATASETS.items():
    print(dataset_name)
    display(make_benchmark_table(dataset_key))


Men H_EURO2024


,Benchmark,Team-games,Hybrid CAs,CAs per team-game,Shot rate (%),Box entry rate (%),Progression toward goal (m)
0,Denmark,4,10,2.50,10.0,40.0,35.3
1,Tournament average,102,333,3.26,18.3,55.6,43.4
2,Semi-finalists/finalists (ESP/FRA/NED/ENG),26,86,3.31,27.9,66.3,43.2
3,Match winners only,34,138,4.06,23.9,62.3,42.5


Women Q_EURO2025


,Benchmark,Team-games,Hybrid CAs,CAs per team-game,Shot rate (%),Box entry rate (%),Progression toward goal (m)
0,Denmark,3,20,6.67,30.0,60.0,42.3
1,Tournament average,62,301,4.85,26.2,58.1,40.2
2,Semi-finalists/finalists (ENG/ITA/GER/ESP),22,111,5.05,27.0,55.9,37.3
3,Match winners only,26,148,5.69,25.0,64.2,36.6


U21 EURO2025


,Benchmark,Team-games,Hybrid CAs,CAs per team-game,Shot rate (%),Box entry rate (%),Progression toward goal (m)
0,Denmark,4,17,4.25,29.4,47.1,36.5
1,Tournament average,62,138,2.23,19.6,50.7,39.5
2,Semi-finalists/finalists (ENG/NED/GER/FRA),22,46,2.09,21.7,58.7,38.9
3,Match winners only,26,69,2.65,21.7,53.6,38.3


## 3. Shot vs. non-shot feature profiles

The feature-profile tables compare counter-attacks that produced a shot with those that did not, and then compare Denmark with the shot-producing profile. The outcome position is defined as the first shot, loss of possession or stoppage within the 20-second analysis window. If no such event is found, the position at the end of the 20-second window is used and marked as window_end_20s in the underlying data.

Outcome progress (m) describes how far up the pitch the attack reached at that outcome point. Outcome distance to goal (m) uses both x and y position when available, while Outcome goal progression speed (m/s) and Duration to outcome (s) are measured over the same start-to-outcome interval.


In [4]:
profile_cols = [
    "label",
    "shot_mean",
    "no_shot_mean",
    "denmark_mean",
    "shot_minus_no_shot_mean",
    "denmark_minus_shot_mean",
]


def make_feature_profile_table(dataset_key):
    table = feature_profiles_by_dataset[dataset_key][profile_cols].copy()
    for col in table.columns[1:]:
        table[col] = table[col].round(2)
    return table.rename(
        columns={
            "label": "Feature",
            "shot_mean": "Shot CAs mean",
            "no_shot_mean": "Non-shot CAs mean",
            "denmark_mean": "Denmark mean",
            "shot_minus_no_shot_mean": "Shot minus non-shot",
            "denmark_minus_shot_mean": "Denmark minus shot profile",
        }
    )


for dataset_key, dataset_name in DATASETS.items():
    print(dataset_name)
    display(make_feature_profile_table(dataset_key))


Men H_EURO2024


,Feature,Shot CAs mean,Non-shot CAs mean,Denmark mean,Shot minus non-shot,Denmark minus shot profile
0,Progression toward goal (m),45.77,42.81,35.31,2.96,-10.47
1,Outcome progress (m),88.82,80.52,75.83,8.30,-12.99
2,Outcome distance to goal (m),19.49,30.30,34.74,-10.81,15.26
3,Outcome goal progression speed (m/s),3.04,3.31,3.38,-0.26,0.33
4,Duration to outcome (s),10.41,11.27,8.46,-0.86,-1.95
5,Directness to goal,0.79,0.85,0.84,-0.07,0.05
6,Time to first action (s),1.95,2.32,2.42,-0.37,0.47
7,Pass count within 20s,1.92,2.42,2.40,-0.50,0.48
8,Touch count within 20s,0.49,0.43,0.30,0.06,-0.19
9,Start progression (m),56.37,42.50,44.99,13.87,-11.38


Women Q_EURO2025


,Feature,Shot CAs mean,Non-shot CAs mean,Denmark mean,Shot minus non-shot,Denmark minus shot profile
0,Progression toward goal (m),44.53,38.67,42.27,5.86,-2.26
1,Outcome progress (m),88.40,84.03,82.90,4.37,-5.50
2,Outcome distance to goal (m),19.43,26.79,24.95,-7.36,5.52
3,Outcome goal progression speed (m/s),3.26,3.78,4.12,-0.52,0.85
4,Duration to outcome (s),8.83,10.56,9.76,-1.74,0.94
5,Directness to goal,0.88,0.90,0.92,-0.02,0.03
6,Time to first action (s),2.09,2.30,2.41,-0.21,0.32
7,Pass count within 20s,1.56,2.14,1.90,-0.59,0.34
8,Touch count within 20s,0.56,0.44,0.55,0.12,-0.01
9,Start progression (m),58.25,43.87,43.20,14.39,-15.05


U21 EURO2025


,Feature,Shot CAs mean,Non-shot CAs mean,Denmark mean,Shot minus non-shot,Denmark minus shot profile
0,Progression toward goal (m),42.02,38.84,36.45,3.18,-5.57
1,Outcome progress (m),88.94,76.19,80.60,12.74,-8.34
2,Outcome distance to goal (m),18.31,34.21,29.30,-15.90,10.99
3,Outcome goal progression speed (m/s),3.81,3.34,2.72,0.47,-1.08
4,Duration to outcome (s),9.33,10.16,10.99,-0.83,1.65
5,Directness to goal,0.83,0.88,0.79,-0.06,-0.03
6,Time to first action (s),1.98,2.13,2.23,-0.15,0.25
7,Pass count within 20s,1.78,2.05,2.06,-0.27,0.28
8,Touch count within 20s,0.70,0.45,0.47,0.25,-0.23
9,Start progression (m),55.00,43.64,48.90,11.36,-6.10
